# Qwen3-ASR Speech Recognition with OpenVINO™

The Qwen3-ASR family includes Qwen3-ASR-1.7B and Qwen3-ASR-0.6B, which support language identification and ASR for 52 languages and dialects. Both leverage large-scale speech training data and the strong audio understanding capability of their foundation model, Qwen3-Omni. Experiments show that the 1.7B version achieves state-of-the-art performance among open-source ASR models and is competitive with the strongest proprietary commercial APIs. Here are the main features:

* **All-in-one**: Qwen3-ASR-1.7B and Qwen3-ASR-0.6B support language identification and speech recognition for 30 languages and 22 Chinese dialects, so as to English accents from multiple countries and regions.

* **Excellent and Fast**: The Qwen3-ASR family ASR models maintains high-quality and robust recognition under complex acoustic environments and challenging text patterns. Qwen3-ASR-1.7B achieves strong performance on both open-sourced and internal benchmarks. While the 0.6B version achieves accuracy-efficient trade-off, it reaches 2000 times throughput at a concurrency of 128. They both achieve streaming / offline unified inference with single model and support transcribe long audio.

* **Novel and strong forced alignment Solution**: We introduce Qwen3-ForcedAligner-0.6B, which supports timestamp prediction for arbitrary units within up to 5 minutes of speech in 11 languages. Evaluations show its timestamp accuracy surpasses E2E based forced-alignment models.

* **Comprehensive inference toolkit**: In addition to open-sourcing the architectures and weights of the Qwen3-ASR series, we also release a powerful, full-featured inference framework that supports vLLM-based batch inference, asynchronous serving, streaming inference, timestamp prediction, and more.

<p align="center">
    <img src="https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-ASR-Repo/overview.jpg" width="100%"/>
<p>

More details can be found in the original [repository](https://github.com/QwenLM/Qwen3-ASR) and [model card](https://huggingface.co/Qwen/Qwen3-ASR-0.6B)

In this tutorial, we will:
1. Install required dependencies
2. Export Qwen3-ASR model to OpenVINO format using Optimum Intel
3. Run inference using OpenVINO
4. Build an interactive Gradio demo for speech recognition

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Select Model](#Select-Model)
- [Export Model to OpenVINO](#Export-Model-to-OpenVINO)
- [Run Inference with OpenVINO](#Run-Inference-with-OpenVINO)
    - [Select Inference Device](#Select-Inference-Device)
    - [Load Model and Run Speech Recognition](#Load-Model-and-Run-Speech-Recognition)
- [Interactive Demo](#Interactive-Demo)


⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO. It may be fully supported and validated in the future.

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/qwen3-asr/qwen3-asr.ipynb" />

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [ ]:
# Fetch notebook_utils module
import requests
from pathlib import Path

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)

if not Path("cmd_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py",
    )
    open("cmd_helper.py", "w").write(r.text)

if not Path("pip_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/pip_helper.py",
    )
    open("pip_helper.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("qwen3-asr.ipynb")

### Install Dependencies
[back to top ⬆️](#Table-of-contents:)

Install required packages for Qwen3-ASR model export and inference with Optimum Intel:

In [ ]:
from pip_helper import pip_install
import platform

pip_install(
    "-q",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
    "torch>=2.6",
    "torchaudio",
    "transformers>=4.53,<5",
    "openvino>=2025.4.0",
    "git+https://github.com/huggingface/optimum-intel.git",
    "qwen-asr",
    "soundfile",
    "librosa",
    "gradio>=4.19",
)

if platform.system() == "Darwin":
    pip_install("numpy<2.0")

## Select Model
[back to top ⬆️](#Table-of-contents:)

Select the Qwen3-ASR model variant to use. The 0.6B model is recommended for faster inference while maintaining good accuracy:

In [ ]:
import ipywidgets as widgets

model_ids = [
    "Qwen/Qwen3-ASR-0.6B",
    "Qwen/Qwen3-ASR-1.7B",
]

model_selector = widgets.Dropdown(
    options=model_ids,
    value=model_ids[0],
    description="Model:",
)

model_selector

## Export Model to OpenVINO
[back to top ⬆️](#Table-of-contents:)

We use [Optimum Intel](https://huggingface.co/docs/optimum/intel/index) to export the Qwen3-ASR model to OpenVINO Intermediate Representation (IR) format using `optimum-cli`. The export handles all model components automatically:

- **Audio Encoder**: Processes mel-spectrogram features into audio embeddings
- **Language Model (Decoder)**: Generates transcription tokens from audio and text embeddings

General command format:

```bash
optimum-cli export openvino --model <model_id_or_path> --trust-remote-code <output_dir>
```

Additionally, you can specify weights compression using `--weight-format` argument with one of following options: `fp32`, `fp16`, `int8` and `int4`. More details about model export provided in [Optimum Intel documentation](https://huggingface.co/docs/optimum/intel/openvino/export#export-your-model).

In [ ]:
from cmd_helper import optimum_cli
from pathlib import Path

model_id = model_selector.value
model_name = model_id.split("/")[-1]
ov_model_dir = Path(f"{model_name}-OV")

if not ov_model_dir.exists():
    optimum_cli(model_id, ov_model_dir, additional_args={"trust-remote-code": "", "weight-format": "fp16"})

## Run Inference with OpenVINO
[back to top ⬆️](#Table-of-contents:)

### Select Inference Device
[back to top ⬆️](#Table-of-contents:)

Select the device for running inference:

In [ ]:
from notebook_utils import device_widget

device = device_widget("CPU", exclude=["NPU"])

device

### Load Model and Run Speech Recognition
[back to top ⬆️](#Table-of-contents:)

Load the exported OpenVINO model using `OVModelForSpeechSeq2Seq` and the processor for audio preprocessing:

In [ ]:
# qwen_asr must be imported to register model with AutoConfig/AutoModel
import qwen_asr  # noqa: F401
from optimum.intel import OVModelForSpeechSeq2Seq
from transformers import AutoProcessor

# Load model
ov_model = OVModelForSpeechSeq2Seq.from_pretrained(
    ov_model_dir,
    trust_remote_code=True,
    device=device.value,
)

# Load processor
processor = AutoProcessor.from_pretrained(ov_model_dir, trust_remote_code=True)

print(f"✅ Model loaded on {device.value}")

### Run Speech Recognition
[back to top ⬆️](#Table-of-contents:)

Let's test the model with a sample audio file:

In [ ]:
import urllib.request
import soundfile as sf
import numpy as np

# Download sample audio
sample_audio_en = Path("sample_en.wav")

if not sample_audio_en.exists():
    print("Downloading English sample audio...")
    urllib.request.urlretrieve(
        "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-ASR-Repo/asr_en.wav",
        sample_audio_en,
    )
    print(f"Downloaded: {sample_audio_en}")

# Load audio
audio_data, sample_rate = sf.read(str(sample_audio_en))

# Resample to 16kHz if needed
target_sr = 16000
if sample_rate != target_sr:
    import librosa

    audio_data = librosa.resample(audio_data, orig_sr=sample_rate, target_sr=target_sr)
    sample_rate = target_sr

# Build text prompt with audio placeholder
text_prompt = processor.apply_chat_template(
    [
        {"role": "system", "content": ""},
        {"role": "user", "content": [{"type": "audio", "audio": ""}]},
    ],
    add_generation_prompt=True,
    tokenize=False,
)

# Process inputs
inputs = processor(
    text=text_prompt,
    audio=audio_data,
    sampling_rate=sample_rate,
    return_tensors="pt",
)

# Generate
print("Running inference...")
generated_ids = ov_model.generate(
    input_features=inputs["input_features"],
    decoder_input_ids=inputs["input_ids"],
    max_new_tokens=256,
)

# Decode - skip the prompt tokens
prompt_len = inputs["input_ids"].shape[1]
generated_only = generated_ids[:, prompt_len:]
transcription = processor.batch_decode(generated_only, skip_special_tokens=True)

print(f"Transcription: {transcription[0]}")

## Interactive Demo
[back to top ⬆️](#Table-of-contents:)

Launch an interactive Gradio demo that allows you to:
- Upload audio files (WAV, MP3, FLAC, etc.)
- Record audio from your microphone
- Select target language or use auto-detection
- View transcription results with inference metrics

The demo is based on the official [Qwen3-ASR Hugging Face Space](https://huggingface.co/spaces/Qwen/Qwen3-ASR).

In [ ]:
from gradio_helper import make_demo

demo = make_demo(ov_model, processor)

# Launch the demo
# If you are launching remotely, specify server_name and server_port:
#   demo.launch(server_name='your_server_name', server_port=7860)
# If you have any issues launching, try:
#   demo.launch(share=True)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)